# Models Experimentation Notebook for IFood Data Science Technical Case

O direcionamento das ofertas corretas para os clientes corretos na hora certa traz grandes impactos para um negócio, uma vez que pode otimizar o uso de recuros, mas, principalmente, fortalecer o engajamento e relacionamento de longo prazo com os clientes.

Existem diferentes formas de endereçarmos e modelarmos esse desafio. Em nossa resolução, utilizaremos uma abordagem de Sistemas de Recomendação, os quais são amplamente utilizados por grandes empresas como Amazon e Netflix.

Existem diferentes estratégias para implementação de sistemas de recomendação, as quais usualmente podem ser classificadas em dois grandes grupos:
- **Content-Based Filtering**: em que o foco são as características/conteúdos dos items e usuários como base para as escolhas. Por exemplo, se um usuário costuma comprar um determinado produto, provavelmente ele também vai ter interesse em outros produtos com características semelhantes. E isso talvez varie com o gênero e faixa etária dos usuários. 
    - **Em nosso caso de uso, poderíamos aproveitar características das ofertas - como os canais de divulgação, valor mínimo, tamanho do desconto, tipo de oferta, duração, etc - e dos usuários - como o gênero, idade, tempo que está registrado na plataforma, etc - para modelar os dados a serem utilizados de base para as recomendações.**
- **Collaborative-Based Filtering**: em que o foco está na interação entre usuários e itens, assim como nas relações em comum entre usuários. Por exemplo, se um usuário semelhante a você comprou um produto X, talvez você também possa gostar desse produto. Daí vem o termo **Colaborativo**. 
    - **Em nosso caso de uso, resolvemos modelar essas relações construindo uma matriz de interação, que diz com quais ofertas cada usuário interagiu. Interagir para a gente significa ter recebido, visto ou completado uma oferta. Além disso, para cada tipo de interação atribuímos um peso: oferta recebida tem peso 1, oferta vista tem peso 3 e oferta completada tem peso 5. Por fim, os valores de cada par (usuário, oferta) está modelado como a média dos pesos dos tipos de interação com a oferta.** 
    - Essa modelagem busca fazer um paralelo com o que vemos, por exemplo, no cenário de avaliações de filmes, em que os valores das interações (usuário, filme) são as notas (ratings) dadas pelos usuários. Em nosso caso, entendemos que um usuário não tem controle direto sobre as **ofertas que recebe**, por isso elas receberam os pesos mais baixos. Todavia, se o usuário resolver **ver a oferta**, isso indica que ele demonstra algum interesse, por isso a pontuação média 3. Por fim, se o usuário utilizou a oferta, é como se ele tivesse dado pontuação máxima, e por isso o 5.

Além desses dois grupos, existem diferentes técnicas para a aplicação de uma **abordagem híbrida**, que combina o melhor dos dois mundos.
Neste case, com o objetivo de se construir uma PoC (Prova de Conceito) simples que demonstre a viabilidade e valor que essa abordagem pode trazer para o problema de direcionamento de ofertas, assim como se aproveitar de algoritmos já implementados no PySpark, vamos empregar a técnica **Alternating Least Squares (ALS)**, que é um método de filtragem colaborativa baseado em modelo.

## 1. Import Libraries

In [1]:
import os
import json
import shutil
import numpy as np
from itertools import chain
import matplotlib.pyplot as plt

from pyspark.sql.functions import col, sum, when, count, to_date, year, avg, coalesce, corr,create_map, lit, desc, size, explode
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder, CrossValidator

# Cria a sessão Spark local
spark = SparkSession.builder.appName("ifoodModelExperimenting").master("local[*]").getOrCreate()

# Verifica
print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/27 22:35:46 WARN Utils: Your hostname, marianna-pinho, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/09/27 22:35:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/27 22:35:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/27 22:35:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


4.0.1


## 2. Utilities

In [3]:
def get_columns_with_nulls(dataframe):
    return dataframe.select([sum(col(c).isNull().cast("int")).alias(c) for c in dataframe.columns])

def info_alternative(dataframe):
    n_rows = dataframe.count()
    print(f"Number of rows: {n_rows}")
    print(f"Number of columns: {len(dataframe.columns)}")
    number_of_nans_in_cols = get_columns_with_nulls(dataframe)
    print("#\tColumn\t\tNon-Null Count\t\tDtype")
    print("---\t------\t\t--------------\t\t-----")
    for index, (col, col_type) in enumerate(dataframe.dtypes):
        print(f"{index}\t{col}\t\t{n_rows - number_of_nans_in_cols.first()[col]}\t\t{col_type}")


def create_id_int_column(data, column_map, original_column_name, new_column_name):
    mapping_expr = create_map([lit(x) for x in chain(*column_map.items())])

    return data.withColumn(
        new_column_name,
        coalesce(
            mapping_expr[col(original_column_name)],
            lit(-1)   # default se não achar a chave
            )
        )

def save_json(data, json_path):
    parent_path = os.path.dirname(json_path)
    os.makedirs(parent_path, exist_ok=True)

    with open(json_path, "w") as file:
        json.dump(data, file)

def load_json(json_path):
    with open(json_path, "r") as file:
        data = json.load(file)
    
    return data

def save_processed_data(data, path_data):
    tmp_path = path_data + "_tmp"
    data.coalesce(1).write.json(tmp_path, mode="overwrite")
    filename = [f for f in os.listdir(tmp_path) if f.startswith("part-")][0]
    shutil.move(os.path.join(tmp_path, filename), path_data)
    shutil.rmtree(tmp_path)

In [4]:
def plot_bars(axes, data_values, data_labels, data_colors, title, x_label, y_label):
    data_colors = data_colors if data_colors else "tab:blue"
    axes.bar(x=data_labels, height=data_values, color=data_colors)
    axes.set_title(title)
    axes.set_xlabel(x_label)
    axes.set_ylabel(y_label)

    return axes

def plot_hist(axes, data_values, n_bins, title, x_label, y_label, data_mean=None, data_median=None, mean_color="tab:red", median_color="tab:green"):

    axes.hist(data_values, bins=n_bins)

    if data_mean is not None:
        axes.axvline(x=data_mean, color=mean_color, linestyle="dashed")
        axes.text(data_mean, axes.get_ylim()[1]-100, "Média", color=mean_color)
    if data_median is not None:
        axes.axvline(x=data_median, color=median_color, linestyle="dashed")
        axes.text(data_median, axes.get_ylim()[1]-200, "Mediana", color=median_color)

    axes.set_title(title)
    axes.set_xlabel(x_label)
    axes.set_ylabel(y_label)

    return axes

## 3. Loading Data

In [5]:
path_user_offer = "ifood-case/data/processed/user_offer_interactions.json"
path_account_id = "ifood-case/data/processed/account_id_map.json"
path_offer_id = "ifood-case/data/processed/offer_id_map.json"

user_offer_interactions = spark.read.json(path_user_offer)
account_id_map = load_json(json_path=path_account_id)
offer_id_map = load_json(json_path=path_offer_id)

print(f"There are {user_offer_interactions.count()} interactions")

There are 63288 interactions


In [6]:
offer_id_map

{'0b1e1539f2cc45b7b9fa7c272da2e1d7': 0,
 '4d5c57ea9a6940dd891ad53e9dbe8da0': 1,
 '9b98b8c7a33c4b65b9aebfe6a799e6d9': 2,
 'f19421c1d4aa40978ebb69ca19b0e20d': 3,
 'fafdcd668e3743c1bb461111dcafc2a4': 4,
 'ae264e3637204a6fb9bb56bc8210ddfd': 5,
 '5a8bc65990b245e5a138643cd4eb9837': 6,
 '2298d6c36e964ae4a3e7e9706d1fb8c2': 7,
 '2906b810c7d4411798c6938adc9daaa5': 8,
 '3f207df678b143eea3cee63160fa8bed': 9}

## 4. Training Models

In [7]:
train_set, test_set = user_offer_interactions.randomSplit([0.8, 0.2])

print(f"There are {train_set.count()} training samples and {test_set.count()} testing samples.")

There are 50617 training samples and 12671 testing samples.


O algoritmo ALS só consegue fazer previsões para usuários e ofertas que ele viu durante o treinamento. Para casos novos, ele retorna NaN.
Conforme vemos abaixo, existem 393 usuários no conjunto de teste que não aparecem no treinamento. Como esse número representa apenas 2% do conjunto de testes, vamos ignorá-los por enquanto. Mas para resultados mais precisos é necessário implementarmos outra estratégia de divisão dos dados que garanta que todos os usuários e itens do teste também estejam no treinamento.

In [8]:
train_users = train_set.select("account_id_int").distinct()
test_users  = test_set.select("account_id_int").distinct()
train_items = train_set.select("offer_id_int").distinct()
test_items  = test_set.select("offer_id_int").distinct()

new_users = test_users.join(train_users, on="account_id_int", how="left_anti")
new_items = test_items.join(train_items, on="offer_id_int", how="left_anti")

print("Usuários no teste que não aparecem no treino:", new_users.count())
print("Ofertas no teste que não aparecem no treino:", new_items.count())


Usuários no teste que não aparecem no treino: 138
Ofertas no teste que não aparecem no treino: 0


A seguir, instanciamos o modelo ALS que vai ficar responsável por, posteriormente, predizer valores de "event_level" para novos pares de (usuário, oferta). Ele utiliza as colunas:
- **account_id_int**: para representar os usuários.
- **offer_id_int**: para representar as ofertas.
- **event_levels**: para representar a força da interação entre usuários e ofertas.

Além disso, vamos experimentar esse modelo com alguns hiperparâmetros:
- **rank**: que determina o tamanho do vetor latente. Vamos experimentá-lo com os valores 13 (quantidade de colunas de features juntando os 3 conjuntos de dados originais) e 20 (teto do intervalo).
- **maxIter**: número máximo de iterações alternadas do modelo durante o treinamento. Vamos testá-lo com os valores 15 e 20 (número comum de iterações).
- **regParam**: coeficiente de regularização da função de custo, usado para reduzir os riscos de overfittings. Vamos testá-lo com valores comumente empregados, como 0,01 e 0,1.

Para avaliar o modelo, vamos utilizar o **RegressionEvaluator** com a métrica de **Root Mean Squared Error (RMSE)**, já que estamos tratando a predição da força de interação entre usuários e ofertas como um problema de regressão.

In [9]:
als_model = ALS(
    userCol="account_id_int",
    itemCol="offer_id_int",
    ratingCol="event_levels",
    coldStartStrategy="drop",
    nonnegative=True
)

param_grid = ParamGridBuilder()\
    .addGrid(als_model.rank, [20])\
    .addGrid(als_model.maxIter, [20])\
    .addGrid(als_model.regParam, [0.1])\
    .build()

evaluator = RegressionEvaluator(metricName="rmse", labelCol="event_levels", predictionCol="prediction")

In [10]:
train_set.cache()

train_valid_split_validator = TrainValidationSplit(
    estimator=als_model,
    estimatorParamMaps=param_grid,
    evaluator=evaluator
)

In [10]:
final_model = train_valid_split_validator.fit(train_set)

25/09/27 22:37:22 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/09/27 22:37:24 ERROR Executor: Exception in task 3.0 in stage 73.0 (TID 445)
java.lang.StackOverflowError
	at java.base/java.io.ObjectInputStream.readClassDesc(ObjectInputStream.java:1900)
	at java.base/java.io.ObjectInputStream.readOrdinaryObject(ObjectInputStream.java:2224)
	at java.base/java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1733)
	at java.base/java.io.ObjectInputStream$FieldValues.<init>(ObjectInputStream.java:2606)
	at java.base/java.io.ObjectInputStream.readSerialData(ObjectInputStream.java:2457)
	at java.base/java.io.ObjectInputStream.readOrdinaryObject(ObjectInputStream.java:2257)
	at java.base/java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1733)
	at java.base/java.io.ObjectInputStream$FieldValues.<init>(ObjectInputStream.java:2606)
	at java.base/java.io.ObjectInputStream.readSerialData(ObjectInputStream.java:2457)
	at java.base/java.

ConnectionRefusedError: [Errno 111] Connection refused

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
model = tvs.fit(training)

In [ ]:
best_model = model.bestModel
best_model

In [ ]:
predictions = best_model.transform(test)

In [ ]:
rmse = evaluator.evaluate(predictions)
rmse

In [ ]:
user_recs = best_model.recommendForAllUsers(3)

In [ ]:
list(filter(lambda key: account_id_map[key] == 1, account_id_map))


In [ ]:
list(filter(lambda key: offer_id_map[key] == 4, offer_id_map))

## 5. Evaluating Models